# 01 — Setup & Data Exploration

This notebook:
1. Sets up the Colab environment (installs deps, downloads data & weights)
2. Explores the competition data (sequences, labels, MSA)

In [ ]:
# === Colab Setup Cell ===
!pip install kaggle -q
import os

# 1. Configure Kaggle API
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file")
    uploaded = files.upload()
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

# 2. Clone code repository
REPO_URL = "https://github.com/YOUR_USER/3drna_cc.git"  # <-- UPDATE THIS
if not os.path.exists('/content/3drna_cc'):
    !git clone {REPO_URL} /content/3drna_cc
%cd /content/3drna_cc

# 3. Install dependencies + download data
from src.setup import setup_environment
setup_environment()

## Explore Training Sequences

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_sequences, load_labels, parse_stoichiometry, get_sequence_lengths

# Load train sequences
train_seq = load_sequences(split='train')
print(f"Training targets: {len(train_seq)}")
print(f"Columns: {list(train_seq.columns)}")
train_seq.head()

In [ ]:
# Sequence length distribution
lengths = train_seq['sequence'].str.len()
print(f"Sequence length: min={lengths.min()}, max={lengths.max()}, "
      f"median={lengths.median():.0f}, mean={lengths.mean():.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(lengths, bins=50, edgecolor='black')
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Count')
axes[0].set_title('Sequence Length Distribution')

axes[1].hist(lengths[lengths < 500], bins=50, edgecolor='black')
axes[1].set_xlabel('Sequence Length')
axes[1].set_title('Sequence Length Distribution (< 500 nt)')
plt.tight_layout()
plt.show()

In [ ]:
# Multi-chain analysis
has_stoich = train_seq['stoichiometry'].str.len() > 0
print(f"Multi-chain targets: {has_stoich.sum()} / {len(train_seq)} "
      f"({100*has_stoich.mean():.1f}%)")

# Ligand analysis
has_ligand = train_seq['ligand_ids'].str.len() > 0
print(f"Targets with ligands: {has_ligand.sum()} / {len(train_seq)} "
      f"({100*has_ligand.mean():.1f}%)")

# Example multi-chain parsing
multi = train_seq[has_stoich].iloc[0]
print(f"\nExample multi-chain target: {multi['target_id']}")
print(f"  Stoichiometry: {multi['stoichiometry']}")
chains = parse_stoichiometry(multi['stoichiometry'], multi['all_sequences'])
for c in chains:
    print(f"  Chain {c['chain_id']}: {len(c['sequence'])} nt x {c['copies']} copies")

## Explore Labels

In [ ]:
# Load training labels
train_labels = load_labels(split='train')
print(f"Label rows: {len(train_labels)}")
print(f"Columns: {list(train_labels.columns)}")
train_labels.head()

In [ ]:
# Coordinate range analysis
for col in ['x_1', 'y_1', 'z_1']:
    vals = train_labels[col].dropna()
    print(f"{col}: min={vals.min():.1f}, max={vals.max():.1f}, "
          f"mean={vals.mean():.1f}, std={vals.std():.1f}")

## Explore MSA

In [ ]:
from src.data.loader import load_msa, get_target_ids
from src.config import MSA_DIR
from pathlib import Path

# Check available MSA files
msa_dir = Path(MSA_DIR) if MSA_DIR else None
target_ids = get_target_ids('train')

msa_depths = []
for tid in target_ids[:50]:  # sample first 50
    msa = load_msa(tid)
    msa_depths.append({'target_id': tid, 'depth': msa['depth']})

msa_df = pd.DataFrame(msa_depths)
print(f"MSA depth stats (first 50 targets):")
print(f"  min={msa_df['depth'].min()}, max={msa_df['depth'].max()}, "
      f"median={msa_df['depth'].median():.0f}")
print(f"  Targets with no MSA: {(msa_df['depth'] == 0).sum()}")

plt.figure(figsize=(10, 4))
plt.hist(msa_df['depth'], bins=30, edgecolor='black')
plt.xlabel('MSA Depth')
plt.ylabel('Count')
plt.title('MSA Depth Distribution')
plt.show()

## Explore Validation Set

In [ ]:
val_seq = load_sequences(split='val')
val_labels = load_labels(split='val')
print(f"Validation targets: {len(val_seq)}")
print(f"Validation label rows: {len(val_labels)}")

val_lengths = val_seq['sequence'].str.len()
print(f"Val sequence length: min={val_lengths.min()}, max={val_lengths.max()}, "
      f"median={val_lengths.median():.0f}")

## Explore Sample Submission

In [ ]:
from src.config import SAMPLE_SUBMISSION

sample_sub = pd.read_csv(SAMPLE_SUBMISSION)
print(f"Submission rows: {len(sample_sub)}")
print(f"Columns: {list(sample_sub.columns)}")

# Count unique targets in submission
sub_targets = sample_sub['ID'].str.rsplit('_', n=1).str[0].unique()
print(f"Unique targets in submission: {len(sub_targets)}")
print(f"Targets: {list(sub_targets)}")
sample_sub.head()